# 05 Validation Diagnostics

## Purpose
This notebook reviews one finished masked language model pretraining run using the saved Hugging Face trainer history.

## What This Notebook Produces
A successful run creates a validation-review folder under `results/validation/` for the selected experiment and saves:

- a training-versus-validation loss plot
- a merged `loss_history.csv`
- a compact `validation_summary.json`
- an updated run-index entry that links back to the validation summary

## Runtime setup

This cell prepares the Colab runtime so the notebook can read experiment artifacts from Google Drive and import the latest shared helper modules from the GitHub repository.

You should expect short status messages showing that Drive was mounted, the repository was cloned or updated, and the repository path was added to Python. If this setup cell fails, the later notebook cells will not be able to load the saved training artifacts or the shared diagnostics helpers.

In [ ]:
# Standard library imports are needed here because the repository helpers are
# not available until after the project repository has been cloned or updated.
import subprocess
import sys
from pathlib import Path

from google.colab import drive

# Mount Google Drive so the notebook can read trainer artifacts and save the
# validation review outputs back to the project folder.
drive.mount('/content/drive')

# Define the public GitHub repository that stores the shared notebook helpers.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = Path('/content') / REPO_NAME

# Clone the repository the first time the notebook runs. If it already exists,
# keep it current so the notebook uses the latest shared code.
if not REPO_DIR.exists():
    print(f'Cloning repository from {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repository already exists at {REPO_DIR}.')

print(f'Updating repository to the latest {GITHUB_REF} changes...')
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_REF],
    check=True,
)

# Add the repository root to the Python path so notebook cells can import
# shared helper modules from the src/ package.
repo_dir_str = str(REPO_DIR)
if repo_dir_str not in sys.path:
    sys.path.insert(0, repo_dir_str)

print(f'Repository ready at: {REPO_DIR}')

## User settings

This is the main cell to review before running the notebook.

The values here identify which completed pretraining experiment should be reviewed and whether previously saved validation-review outputs may be replaced. Everything else in the notebook is derived from these settings, so this is the only place where you should normally need to edit paths or run-selection values.


In [ ]:
from pathlib import Path

# Update PROJECT_ROOT if your Google Drive project folder uses a different name
# or location. This is the main path value to verify before running.
PROJECT_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# Choose the tokenizer family and the exact experiment folder to review.
TOKENIZER_FAMILY = 'manual'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2'

# Set this to True only when you intentionally want to replace previously
# saved validation-review outputs for the selected experiment.
OVERWRITE_EXISTING_OUTPUTS = False


## Validate the selected run and load its metadata

This cell confirms that the selected checkpoint folder, trainer-state file, experiment-metadata file, and run index all exist. It also creates the validation results folder if needed, checks the shared overwrite policy for the files this notebook plans to save, and loads the saved experiment metadata so later summary files stay linked to the original run settings.

You should expect the checkpoint directory and validation results directory to print. If a path is wrong or overwrite protection blocks existing outputs, this cell is designed to fail early before any plotting or summary work begins.


In [ ]:
from src.training_diagnostics import prepare_validation_run

# Build the validated run context that later cells reuse for path access,
# saved metadata, validation-output locations, and overwrite protection.
run_context = prepare_validation_run(
    project_root=PROJECT_ROOT,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=EXPERIMENT_NAME,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
)

validation_paths = run_context['paths']
experiment_metadata = run_context['experiment_metadata']

print(f"Checkpoint directory: {validation_paths['checkpoint_dir']}")
print(f"Validation results directory: {validation_paths['validation_results_dir']}")
print(f'Overwrite existing outputs: {OVERWRITE_EXISTING_OUTPUTS}')


## Load the trainer history

This cell loads the saved Hugging Face trainer log history and separates it into training-loss rows and validation-loss rows.

The displayed previews help confirm that both kinds of history are present and that the epoch values look sensible. If either training or validation rows are missing, the notebook stops here because the later diagnostics would not be meaningful.

In [ ]:
from IPython.display import display

from src.training_diagnostics import load_trainer_history, split_train_eval_history

# Load the saved trainer history and separate it into the two tables used by
# the later plotting and validation-summary steps.
log_history_df = load_trainer_history(validation_paths['trainer_state_path'])
train_rows, eval_rows = split_train_eval_history(log_history_df)

if train_rows.empty or eval_rows.empty:
    raise ValueError(
        'Trainer history does not contain both training and validation loss rows.'
    )

print(f'Training-history rows: {len(train_rows):,}')
print(f'Validation-history rows: {len(eval_rows):,}')

display(train_rows.head())
display(eval_rows.head())

## Plot training and validation loss

This cell displays the main validation-diagnostics plot and saves a copy to the validation results folder.

You should expect a line plot with epoch on the x-axis and loss on the y-axis. The key thing to look for is whether validation loss improved steadily, flattened out, or started rising while training loss kept falling.

In [ ]:
from src.training_diagnostics import plot_loss_curves, save_loss_curve_plot

# Display the loss curves inline so the run can be reviewed visually inside the
# notebook before the saved summary files are written.
plot_loss_curves(train_rows, eval_rows)
save_loss_curve_plot(train_rows, eval_rows, validation_paths['loss_plot_path'])

print(f"Loss plot saved to: {validation_paths['loss_plot_path']}")

## Summarize validation behavior

This cell builds the compact validation summary used for later review. It reports the best validation epoch, the best validation loss, the final validation loss, and a conservative recommendation about whether a continuation run is worth considering.

The table should be easy to scan. A best epoch very late in training, especially without obvious recent deterioration, is the main pattern that supports trying a continuation run.

In [ ]:
from IPython.display import display

from src.training_diagnostics import build_validation_review

# Build the merged loss-history export, the summary JSON payload, and the small
# table that highlights the most important validation-review outcomes.
review_bundle = build_validation_review(
    run_context=run_context,
    train_rows=train_rows,
    eval_rows=eval_rows,
)

loss_history_export = review_bundle['loss_history_export']
validation_summary = review_bundle['validation_summary']
continuation_recommendation = review_bundle['continuation_recommendation']
summary_df = review_bundle['summary_df']

display(summary_df)

## Save the validation review and update the run index

This cell writes the merged loss history and validation summary to the validation results folder, then updates the shared run index so the review is easy to find later.

The printed paths confirm where the saved review files ended up. The continuation recommendation is also written into the run-index notes field so it stays attached to the reviewed run.

In [ ]:
from src.training_diagnostics import save_validation_review

# Save the validation-review artifacts and register their paths in the shared
# run index for easier lookup later in the project workflow.
saved_paths = save_validation_review(
    run_context=run_context,
    validation_summary=validation_summary,
    loss_history_export=loss_history_export,
    notes=continuation_recommendation,
)

print(f"Validation summary saved to: {saved_paths['validation_summary_path']}")
print(f"Loss history CSV saved to: {saved_paths['loss_history_path']}")